## Импорты

In [1]:
CONFIG_NAME = "whisper.yaml"
#CONFIG_NAME = "custom_lstm_correction.yaml"
#CONFIG_NAME = "custom_CNN_RNN.yaml"

In [2]:
import sys
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pathlib import Path

# Допустим, что ноутбук находится в той же директории, что и папка acoustic/
sys.path.insert(0, str(Path.cwd()))

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers.generation_utils").setLevel(logging.ERROR)

In [3]:
from acoustic.utils.config import load_config
from acoustic.dataset.load_dataset import load_and_prepare_dataset
from acoustic.models.load_model import build_model
from acoustic.training.load_metrics import load_metrics
from acoustic.training.callbacks import get_callback
from acoustic.training import get_trainer_class


## Загрузка конфигурации

In [4]:
CONFIG_PATH = f"acoustic/configs/{CONFIG_NAME}"

cfg = load_config(CONFIG_PATH, overrides=None)

print("Configuration loaded")

Configuration loaded


## Загрузка датасета

In [5]:
dataset = load_and_prepare_dataset(cfg)

## Инициализация модели

In [6]:
model, processor, data_collator = build_model(cfg)

print("Model built")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model built


## Создание и загрузка метрик, callbacks, trainer

In [7]:
metrics_list = load_metrics(cfg['training']['metrics'])
print(f"Metrics: {cfg['training']['metrics']}")

callbacks = []
for cb_name in cfg['training']['callbacks']:
    callbacks.append(get_callback(cb_name))

Metrics: ['wer', 'cer', 'f1', 'detailed_stats', 'ser', 'space_wer']


In [8]:

trainer_name = cfg['training'].get('trainer', 'BaseTrainer')
TrainerClass = get_trainer_class(trainer_name)

trainer = TrainerClass(
    cfg=cfg,
    model=model,
    processor=processor,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    metrics=metrics_list,
    callbacks=callbacks,
    data_collator=data_collator
)

## Обучение

In [9]:
print("Starting training")
trainer.train()

Starting training


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
Training: 100%|██████████| 800/800 [00:00<?, ?step/s]
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


{'train_runtime': 0.4651, 'train_samples_per_second': 27532.134, 'train_steps_per_second': 1720.086, 'train_loss': 0.0, 'epoch': 4.99}


## Проверка

In [10]:
print("Demo on validation examples")
import random
import torch
from acoustic.models import get_generate_method

eval_dataset = dataset['validation']

if eval_dataset and len(eval_dataset) > 0:
    indices = random.sample(range(len(eval_dataset)), min(10, len(eval_dataset)))
    device = next(model.parameters()).device
    model.eval()
    builder_key = cfg['model']['builder']
    generate_fn = get_generate_method(builder_key)

    
    for i in indices:
        example = eval_dataset[i]

        # Ветка для текстового корректора
        if builder_key == "correction_model":
            stt_text = "исправь: " + example["stt_text"]
            ref_text = example["reference"]

            inputs = processor(stt_text, return_tensors="pt", truncation=True, padding=True, max_length=128)
            input_data = inputs["input_ids"].to(device)

            with torch.no_grad():
                predicted_ids = generate_fn(model, input_data, processor)
            pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            print(f"\nExample {i+1}:")
            print(f" ASR output : {stt_text}")
            print(f" Corrected  : {pred_text}")
            print(f" Reference  : {ref_text}")

        elif builder_key == "lstm_correction":
            stt_text = example["stt_text"]
            ref_text = example["reference"]

            inputs = processor(stt_text, return_tensors="pt", truncation=True, padding=True, max_length=220)
            input_data = inputs["input_ids"].to(device)

            with torch.no_grad():
                outputs = model(input_data)
            pred_ids = outputs["logits"]
            pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

            print(f"\nExample {i+1}:")
            print(f" ASR output : {stt_text}")
            print(f" Corrected  : {pred_text}")
            print(f" Reference  : {ref_text}")

        else:
            audio_array = example["audio"]["array"]
            ref_text = example["sentence"]

            inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
            input_key = "input_features" if "input_features" in inputs else "input_values"
            input_data = inputs[input_key].to(device)

            with torch.no_grad():
                predicted_ids = generate_fn(model, input_data, processor)
            pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            print(f"\nExample {i+1}:")
            print(f" Reference: {ref_text}")
            print(f" Prediction: {pred_text}")
else:
    print("No validation dataset for demo.")

Demo on validation examples

Example 328:
 Reference: амазонка также является самой широкой рекой на земле ширина которой иногда составляет шесть миль
 Prediction: амазонка также является самой широкой рекой на земле ширина которой иногда составляет 6 миль

Example 58:
 Reference: однако большинство знаков указываются только на каталанском языке поскольку он по закону является первым официальным языком
 Prediction: однако большинство знаков указываются только на каталандском языке поскольку он по закону является первым официальным языком

Example 13:
 Reference: как только вы выйдете из течения плыть обратно будет не труднее чем обычно
 Prediction: как только вы выйдете из течения плыть обратно будет не труднее чем обычно

Example 141:
 Reference: баба шьям были поданы 108 тарелок чаппан-бхог в индуизме 56 различных яств включая сладости фрукты орехи и другие блюда преподносимые божеству
 Prediction: бабашьям были поданы 108 тарелок чапанбхог в индуизме 56 различных яств включая сладос